<a href="https://colab.research.google.com/github/AkankshaB123/python/blob/main/AdCampaignOptimisation_Simulator_LinearProgramming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import pandas as pd
# import numpy as np
# from pulp import *

# df = pd.read_csv(csv_path)
# n = len(df)

# # Create a MILP problem
# problem = LpProblem("Simple MILP Problem", LpMaximize)

# # Define decision variables
# x = LpVariable.dicts("x", range(n), lowBound=0, cat='Integer')
# y = LpVariable.dicts('y', range(n), cat='Binary')

# # Objective
# problem += lpSum(x[i]*df['gain_loss'][i] for i in range(n))

# # constraints

# # limit to amount of land available
# problem += lpSum(x[i]*df['square_feet'][i]*1.25 for i in range(n)) <= 150000

# # requirements for diversity in home sizes
# problem += lpSum(x[i]*df['small_house'][i] for i in range(n)) >= 15
# problem += lpSum(x[i]*df['medium_house'][i] for i in range(n)) >= 15
# problem += lpSum(x[i]*df['large_house'][i] for i in range(n)) >= 10

# # Create at least 6 unique floorplans
# for i in range(n):
#     # if x is 0, y has to be 0
#     problem += x[i] >= y[i]

# # if x = 1, y coud be 0 or 1
# # but because we want sum(y) to be >= 6, the optimization
# # will assign y to be 1
# problem += lpSum(y[i] for i in range(n)) >= 6

# # solve problem
# problem.solve()

# # print solution
# for i in range(n):
#     print(f'{i + 1} : {value(x[i])}')

# # print optimal profit
# print("Optimal profit :", value(problem.objective))

### Ad Campaign Optimization Simulator

This simulator demonstrates how to use Mixed-Integer Linear Programming (MILP) to optimize ad campaign spending. The goal is to maximize total expected revenue while adhering to a daily budget and ensuring diverse product exposure.

**Scenario:** You have a daily ad budget and a list of products, each with an expected revenue per impression and a cost per impression. You need to decide how many impressions to allocate to each product.

**Constraints include:**
*   Staying within the `TOTAL_BUDGET`.
*   Ensuring that if a product is advertised, it receives a `MIN_IMPRESSIONS_IF_ADVERTISED`.
*   Advertising at least a `MIN_UNIQUE_PRODUCTS` across the entire campaign.
*   Meeting minimum advertising requirements for specific product categories (e.g., `MIN_ELECTRONICS_PRODUCTS`, `MIN_CLOTHING_PRODUCTS`).

In [2]:
import pandas as pd
import numpy as np
from pulp import *

# --- Parameters for the Ad Campaign Simulator ---
TOTAL_BUDGET = 5000  # Total daily ad budget
MIN_IMPRESSIONS_IF_ADVERTISED = 100 # Minimum impressions for an advertised product
MIN_UNIQUE_PRODUCTS = 3 # At least this many unique products must be advertised
MIN_ELECTRONICS_PRODUCTS = 1 # At least this many electronics products must be advertised
MIN_CLOTHING_PRODUCTS = 1 # At least this many clothing products must be advertised
BIG_M = 1000000 # A sufficiently large number for Big-M formulation in MILP

# --- Create a sample DataFrame for products ---
data = {
    'product_id': [f'P{i+1}' for i in range(10)],
    'category': ['Electronics', 'Clothing', 'Books', 'Electronics', 'Home Goods',
                 'Clothing', 'Books', 'Electronics', 'Home Goods', 'Clothing'],
    'expected_revenue_per_impression': [0.005, 0.003, 0.002, 0.006, 0.004,
                                        0.004, 0.003, 0.007, 0.003, 0.005],
    'cost_per_impression': [0.001, 0.0008, 0.0005, 0.0012, 0.0009,
                            0.001, 0.0006, 0.0015, 0.0007, 0.0011]
}
df_ads = pd.DataFrame(data)
n_ads = len(df_ads)

# Display the sample product data
print("Sample Product Data:")
display(df_ads)

# --- Create the MILP problem for Ad Campaign Optimization ---
problem_ads = LpProblem("Ad Campaign Optimization", LpMaximize)

# --- Define decision variables ---
# impressions[i]: Integer, number of impressions for product i
impressions = LpVariable.dicts("impressions", range(n_ads), lowBound=0, cat='Integer')
# is_active[i]: Binary, 1 if product i is advertised, 0 otherwise
is_active = LpVariable.dicts('is_active', range(n_ads), cat='Binary')

# --- Objective Function: Maximize total expected revenue ---
problem_ads += lpSum(impressions[i] * df_ads['expected_revenue_per_impression'][i] for i in range(n_ads)), "Total Expected Revenue"

# --- Constraints ---

# 1. Total Budget Constraint: Total cost of impressions must not exceed the budget
problem_ads += lpSum(impressions[i] * df_ads['cost_per_impression'][i] for i in range(n_ads)) <= TOTAL_BUDGET, "Total Budget"

# 2. Link impressions and is_active variables
for i in range(n_ads):
    # If is_active[i] is 1, impressions[i] must be at least MIN_IMPRESSIONS_IF_ADVERTISED
    problem_ads += impressions[i] >= is_active[i] * MIN_IMPRESSIONS_IF_ADVERTISED, f"Min Impressions for Product {i}"
    # If is_active[i] is 0, impressions[i] must be 0 (or effectively 0 due to the upper bound)
    # If is_active[i] is 1, impressions[i] can be up to BIG_M (a large number)
    problem_ads += impressions[i] <= is_active[i] * BIG_M, f"Max Impressions for Product {i}"

# 3. Minimum Unique Products Advertised: Ensure at least a certain number of unique products are active
problem_ads += lpSum(is_active[i] for i in range(n_ads)) >= MIN_UNIQUE_PRODUCTS, "Minimum Unique Products"

# 4. Category Diversity Constraints: Ensure minimum products from specific categories are active
problem_ads += lpSum(is_active[i] for i in range(n_ads) if df_ads['category'][i] == 'Electronics') >= MIN_ELECTRONICS_PRODUCTS, "Min Electronics Products"
problem_ads += lpSum(is_active[i] for i in range(n_ads) if df_ads['category'][i] == 'Clothing') >= MIN_CLOTHING_PRODUCTS, "Min Clothing Products"

# --- Solve the problem ---
print("\nSolving the Ad Campaign Optimization Problem...")
problem_ads.solve()

# --- Print Solution ---
print("Status:", LpStatus[problem_ads.status])
print("\n--- Ad Campaign Results ---")
total_cost = 0
total_revenue = 0
active_products_count = 0
results = []

for i in range(n_ads):
    if value(is_active[i]) == 1: # If the product is active
        active_products_count += 1
        product_impressions = value(impressions[i])
        cost = product_impressions * df_ads['cost_per_impression'][i]
        revenue = product_impressions * df_ads['expected_revenue_per_impression'][i]
        total_cost += cost
        total_revenue += revenue
        results.append({
            'Product': df_ads['product_id'][i],
            'Category': df_ads['category'][i],
            'Impressions Allocated': product_impressions,
            'Ad Cost': f'${cost:.2f}',
            'Expected Revenue': f'${revenue:.2f}'
        })

results_df = pd.DataFrame(results)
if not results_df.empty:
    display(results_df)
else:
    print("No products were advertised based on the given constraints.")

print(f"\nTotal Products Advertised: {active_products_count}")
print(f"Total Ad Spend: ${total_cost:.2f}")
print(f"Total Expected Revenue: ${total_revenue:.2f}")
print(f"Optimal Profit (Expected Revenue - Ad Cost): ${total_revenue - total_cost:.2f}")

Sample Product Data:


,product_id,category,expected_revenue_per_impression,cost_per_impression
0,P1,Electronics,0.005,0.0010
1,P2,Clothing,0.003,0.0008
2,P3,Books,0.002,0.0005
3,P4,Electronics,0.006,0.0012
4,P5,Home Goods,0.004,0.0009
5,P6,Clothing,0.004,0.0010
6,P7,Books,0.003,0.0006
7,P8,Electronics,0.007,0.0015
8,P9,Home Goods,0.003,0.0007
9,P10,Clothing,0.005,0.0011



Solving the Ad Campaign Optimization Problem...
Status: Optimal

--- Ad Campaign Results ---


/usr/local/lib/python3.12/dist-packages/pulp/pulp.py:1706: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


,Product,Category,Impressions Allocated,Ad Cost,Expected Revenue
0,P1,Electronics,1000000.0,$1000.00,$5000.00
1,P4,Electronics,1000000.0,$1200.00,$6000.00
2,P7,Books,1000000.0,$600.00,$3000.00
3,P8,Electronics,999999.0,$1500.00,$6999.99
4,P10,Clothing,636365.0,$700.00,$3181.83



Total Products Advertised: 5
Total Ad Spend: $5000.00
Total Expected Revenue: $24181.82
Optimal Profit (Expected Revenue - Ad Cost): $19181.82


**Remaining Products** (based on the rules we have set above)
1. Lower Efficiency: lower expected_rpi and cpi
2. Budget Limitations: Total_Budget
3. Constraints: Minimum Electronics, Clothing, and minimum unique products

In [ ]:
# Ad Cost for P1: 1,000,000 impressions * $0.001/impression = $1,000.00
# Expected Revenue for P1: 1,000,000 impressions * $0.005/impression = $5,000.00